# SQLite

[Video](https://youtu.be/Y5zdZpHPjvo)

Wir arbeiten mit der normalisierten Datenbank zur LK-Wahl aus dem Datenbank-Kapitel:

Lehrer(<u>Kuerzel</u>, Lehrer) <br>
Kurse(<u>KID</u>, Kuerzel↑, Stunden) <br>
Schueler(<u>SNr</u>, Name, Vorname) <br>
LKWahl(<u>SNr↑</u>, <u>KID↑</u>)



<table style="border-collapse: separate; border-spacing: 0; margin-left:0; margin-right:auto;">
<tr valign="top">

<td>
<table>
<tr>
    <th colspan="2">LKWahl</th>
</tr>
<tr>
    <th>SNr</th>
    <th>KID</th>
</tr>
<tr><td>120</td><td>M1</td></tr>
<tr><td>120</td><td>Ph1</td></tr>
<tr><td>121</td><td>Bio1</td></tr>
<tr><td>121</td><td>Ph1</td></tr>
<tr><td>122</td><td>M2</td></tr>
<tr><td>122</td><td>Ge2</td></tr>
</table>
</td>

<td style="padding-left:80px;">
<table>
<tr>
    <th colspan="3">Schueler</th>
</tr>
<tr>
    <th>SNr</th>
    <th>Name</th>
    <th>Vorname</th>
</tr>
<tr><td>120</td><td>Riedberg</td><td>Malte</td></tr>
<tr><td>121</td><td>Dreis</td><td>Lena</td></tr>
<tr><td>122</td><td>Rödding</td><td>Maike</td></tr>
</table>
</td>

<td style="padding-left:80px;">
<table>
<tr>
    <th colspan="3">Kurse</th>
</tr>
<tr>
    <th>KID</th>
    <th>Kuerzel</th>
    <th>Stunden</th>
</tr>
<tr><td>M1</td><td>SC</td><td>5</td></tr>
<tr><td>Ph1</td><td>SC</td><td>5</td></tr>
<tr><td>Bio1</td><td>DA</td><td>5</td></tr>
<tr><td>M2</td><td>AD</td><td>5</td></tr>
<tr><td>Ge2</td><td>GE</td><td>5</td></tr>
</table>
</td>

<td style="padding-left:80px;">
<table>
<tr>
    <th colspan="2">Lehrer</th>
</tr>
<tr>
    <th>Kuerzel</th>
    <th>Lehrer</th>
</tr>
<tr><td>SC</td><td>Schulte</td></tr>
<tr><td>DA</td><td>Dallmeyer</td></tr>
<tr><td>AD</td><td>Adam</td></tr>
<tr><td>GE</td><td>Gebauer</td></tr>
</table>
</td>

</tr>
</table>

## SQ 1 · SQLite: Datenbank anlegen

### Inhalte

Wir verwenden das DBMS **SQLite**. Python bietet mit dem Modul `sqlite3` eine **Datenbankschnittstelle**, über die wir unsere Anweisungen an das DBMS übermitteln. **SQL** ist die Sprache, in der diese Anweisungen formuliert werden. Die ganze Datenbank steht bei SQLite in einer einzigen Datei, hier `lkwahl.db`.

##### Connection und Cursor

Zuerst bauen wir eine **Verbindung** (Connection) zur Datenbank auf. Über einen **Cursor** übermitteln wir dann die SQL-Anweisungen mit `execute()`. Jede Arbeit mit der Datenbank folgt diesem Muster:

In [ ]:
import sqlite3
con = sqlite3.connect("lkwahl.db")    # wird neu angelegt, falls sie noch nicht existiert
cur = con.cursor()

# hier kommen unsere SQL-Anweisungen hin

con.commit()        # falls Änderungen in der Datenbank gespeichert werden sollen
con.close()

`commit()` brauchen wir nur, wenn wir die Datenbank **verändern**. Ohne `commit()` gehen die Änderungen beim Schließen verloren.

##### DROP TABLE und CREATE TABLE

Mit `CREATE TABLE` legen wir eine Tabelle an. Für jede Spalte geben wir einen Namen und einen Datentyp an:

| SQL-Datentyp | entspricht in Python |
|:--|:--|
| `INTEGER` | `int` |
| `REAL` | `float` |
| `TEXT` | `str` |

Zusätzlich können wir angeben:

- `PRIMARY KEY` – die Spalte ist der Primärschlüssel. Besteht er aus mehreren Spalten, schreiben wir `PRIMARY KEY (Spalte1, Spalte2)` ans Ende.
- `NOT NULL` – die Spalte darf nicht leer bleiben.
- `FOREIGN KEY (Spalte) REFERENCES Tabelle(Spalte)` – die Spalte ist ein Fremdschlüssel.

Mit `DROP TABLE IF EXISTS` löschen wir eine Tabelle, falls es sie schon gibt. So können wir die Zelle mehrmals ausführen.

In [ ]:
import sqlite3
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("DROP TABLE IF EXISTS Lehrer")
cur.execute("""
CREATE TABLE Lehrer (
    Kuerzel TEXT PRIMARY KEY,
    Lehrer TEXT NOT NULL
)
""")

con.commit()
con.close()

##### Die Datenbank LKWahl aufbauen

Wir legen jetzt die vier Tabellen der normalisierten LK-Wahl an und füllen sie mit `INSERT INTO` (mehr dazu in SQ 2). `execute()` kann immer nur **eine** SQL-Anweisung ausführen. Mit `executescript()` können wir mehrere Anweisungen auf einmal übergeben. Sie werden durch einen **Strichpunkt** getrennt.

Führe diese Zelle aus, bevor du mit SQ 2 weitermachst. Sie stellt auch jederzeit den Ausgangszustand wieder her.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.executescript("""
DROP TABLE IF EXISTS LKWahl;
DROP TABLE IF EXISTS Schueler;
DROP TABLE IF EXISTS Kurse;
DROP TABLE IF EXISTS Lehrer;

CREATE TABLE Lehrer (
    Kuerzel TEXT PRIMARY KEY,
    Lehrer TEXT NOT NULL
);

CREATE TABLE Kurse (
    KID TEXT PRIMARY KEY,
    Kuerzel TEXT NOT NULL,
    Stunden INTEGER,
    FOREIGN KEY (Kuerzel) REFERENCES Lehrer(Kuerzel)
);

CREATE TABLE Schueler (
    SNr INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Vorname TEXT NOT NULL
);

CREATE TABLE LKWahl (
    SNr INTEGER,
    KID TEXT,
    PRIMARY KEY (SNr, KID),
    FOREIGN KEY (SNr) REFERENCES Schueler(SNr),
    FOREIGN KEY (KID) REFERENCES Kurse(KID)
);

INSERT INTO Lehrer VALUES ('SC', 'Schulte');
INSERT INTO Lehrer VALUES ('DA', 'Dallmeyer');
INSERT INTO Lehrer VALUES ('AD', 'Adam');
INSERT INTO Lehrer VALUES ('GE', 'Gebauer');

INSERT INTO Kurse VALUES ('M1', 'SC', 5);
INSERT INTO Kurse VALUES ('Ph1', 'SC', 5);
INSERT INTO Kurse VALUES ('Bio1', 'DA', 5);
INSERT INTO Kurse VALUES ('M2', 'AD', 5);
INSERT INTO Kurse VALUES ('Ge2', 'GE', 5);

INSERT INTO Schueler VALUES (120, 'Riedberg', 'Malte');
INSERT INTO Schueler VALUES (121, 'Dreis', 'Lena');
INSERT INTO Schueler VALUES (122, 'Rödding', 'Maike');

INSERT INTO LKWahl VALUES (120, 'M1');
INSERT INTO LKWahl VALUES (120, 'Ph1');
INSERT INTO LKWahl VALUES (121, 'Bio1');
INSERT INTO LKWahl VALUES (121, 'Ph1');
INSERT INTO LKWahl VALUES (122, 'M2');
INSERT INTO LKWahl VALUES (122, 'Ge2');
""")

con.commit()
con.close()
print('Datenbank erfolgreich erstellt.')

##### Den Inhalt einer Tabelle ansehen

Um zu prüfen, was in einer Tabelle steht, schicken wir eine **Abfrage** an das DBMS. `SELECT * FROM Lehrer` bedeutet: Gib alle Spalten (`*`) aller Datensätze der Tabelle *Lehrer* zurück. Mit `cur.fetchall()` holen wir das Ergebnis ab. Es ist eine Liste, in der jeder Datensatz ein Tupel ist. Mit einer Schleife geben wir die Datensätze nacheinander aus.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Lehrer
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Dieses Muster verwenden wir ab jetzt, um uns Tabellen anzusehen. Statt `Lehrer` können wir jede andere Tabelle einsetzen. Wie man mit `SELECT` gezielt Spalten und Datensätze auswählt, lernen wir in SQ 3.

### Aufgabe 1

Welchen SQL-Datentyp wählst du für die folgenden Attribute? Begründe besonders bei C und D.

```
A)  Preis eines Artikels, z. B. 12.99       C)  Postleitzahl, z. B. 01067
B)  Anzahl der Seiten eines Buches          D)  Telefonnummer, z. B. 0731 123456
```

#### Lösung

| | Attribut | Datentyp | Begründung |
|:--|:--|:--|:--|
| A | Preis | `REAL` | Dezimalzahl |
| B | Seitenzahl | `INTEGER` | ganze Zahl, mit der man auch rechnen kann |
| C | Postleitzahl | `TEXT` | Als Zahl würde die führende Null verloren gehen: 01067 würde zu 1067. Mit einer PLZ rechnet man außerdem nicht. |
| D | Telefonnummer | `TEXT` | Führende Null und Leerzeichen bleiben erhalten. Auch hier rechnet man nicht. |

**Faustregel:** Nicht alles, was aus Ziffern besteht, ist eine Zahl. Wenn man nicht damit rechnet, ist `TEXT` meist die richtige Wahl.

### Aufgabe 2

Lege in der Datenbank `bibliothek.db` die Tabelle *Buch* an. Die BuchID ist der Primärschlüssel, der Titel darf nicht leer sein.

-  Buch(<u>BuchID</u>,Titel,Autor,Erscheinungsjahr)

Prüfe danach mit der folgenden Abfrage, welche Tabellen es in der Datenbank gibt. Gib das Ergebnis wie oben mit `cur.fetchall()` und einer Schleife aus. `sqlite_master` ist eine Tabelle, in der SQLite selbst Buch über die Datenbank führt.

```
SELECT name FROM sqlite_master WHERE type = 'table'
```

#### Lösung

In [ ]:
import sqlite3

con = sqlite3.connect("bibliothek.db")
cur = con.cursor()

cur.execute("DROP TABLE IF EXISTS Buch")
cur.execute("""
CREATE TABLE Buch (
    BuchID INTEGER PRIMARY KEY,
    Titel TEXT NOT NULL,
    Autor TEXT,
    Erscheinungsjahr INTEGER
)
""")
con.commit()

cur.execute("SELECT name FROM sqlite_master WHERE type = 'table'")
for datensatz in cur.fetchall():
    print(datensatz)        # ('Buch',)

con.close()

## SQ 2 · Daten einfügen, ändern und löschen

### Inhalte

Für Änderungen an den Daten gibt es drei SQL-Anweisungen. Nach jeder Änderung brauchen wir `commit()`.

| Anweisung | Wirkung | Beispiel |
|:--|:--|:--|
| `INSERT INTO` | Datensatz einfügen | `INSERT INTO Lehrer VALUES ('MU', 'Mueller')` |
| `UPDATE … SET … WHERE` | Datenfelder ändern | `UPDATE Lehrer SET Lehrer = 'Müller' WHERE Kuerzel = 'MU'` |
| `DELETE FROM … WHERE` | Datensätze löschen | `DELETE FROM Lehrer WHERE Kuerzel = 'MU'` |

**Vorsicht:** Bei `UPDATE` und `DELETE` legt `WHERE` fest, **welche** Datensätze betroffen sind. Ohne `WHERE` werden **alle** Datensätze geändert bzw. gelöscht!

Wir fügen eine neue Lehrerin ein, korrigieren ihren Namen und löschen sie wieder. Nach jedem Schritt sehen wir uns die Tabelle an.

##### INSERT INTO: Daten einfügen

Die Werte stehen in der Reihenfolge der Spalten. Strings stehen in einfachen Hochkommata.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("INSERT INTO Lehrer VALUES ('MU', 'Mueller')")

con.commit()
con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Lehrer
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

##### UPDATE: Daten ändern

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("UPDATE Lehrer SET Lehrer = 'Müller' WHERE Kuerzel = 'MU'")

con.commit()
con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Lehrer
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

##### DELETE FROM: Daten löschen

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("DELETE FROM Lehrer WHERE Kuerzel = 'MU'")

con.commit()
con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Lehrer
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

##### Referentielle Integrität in SQLite

SQLite prüft Fremdschlüssel nur, wenn wir es mit `PRAGMA foreign_keys = ON` einschalten. Dann verweigert es zum Beispiel das Löschen von Dallmeyer, weil der Kurs Bio1 noch auf das Kürzel DA verweist (vgl. das Kapitel *Referentielle Integrität*). Das DBMS meldet einen **IntegrityError**, und die Datenbank bleibt unverändert.

**Wichtig:** Bricht eine Zelle zwischen `connect()` und `close()` mit einem Fehler ab, bleibt die Verbindung offen. Unter Windows lässt sich die Datenbankdatei dann nicht löschen oder verschieben, bis wir `con.close()` ausführen oder den Kernel neu starten (**Kernel → Restart Kernel**).

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("PRAGMA foreign_keys = ON")
cur.execute("DELETE FROM Lehrer WHERE Kuerzel = 'DA'")
con.commit()

con.close()

In [ ]:
# Die Zelle oben bricht mit dem IntegrityError ab, bevor con.close() erreicht wird.
# Diese Zelle schließt die Verbindung, damit die Datei nicht gesperrt bleibt.
con.close()

##### Anmerkung:

PRAGMA ist ein Wort aus dem Griechischen und heißt „Sache“ oder „Handlung“. In der Informatik steht es seit Langem für eine Anweisung an das System selbst. Ein Pragma verarbeitet also keine Daten, sondern stellt ein, wie das System arbeiten soll. 

In SQLite ist PRAGMA ein eigener Befehl, mit dem man interne Einstellungen des DBMS abfragt oder ändert. Er gehört nicht zum SQL-Standard, andere Datenbanksysteme wie MySQL oder PostgreSQL haben dafür andere Befehle.

### Aufgabe 1

Die folgende Zelle legt die Datenbank `bibliothek.db` mit den drei Büchern aus dem Kapitel *Tabellen* neu an. Führe sie aus.

In [ ]:
import sqlite3

con = sqlite3.connect("bibliothek.db")
cur = con.cursor()

cur.executescript("""
DROP TABLE IF EXISTS Buch;
CREATE TABLE Buch (
    BuchID INTEGER PRIMARY KEY,
    Titel TEXT NOT NULL,
    Autor TEXT,
    Erscheinungsjahr INTEGER
);
INSERT INTO Buch VALUES (101, 'Die Physiker', 'Dürrenmatt', 1962);
INSERT INTO Buch VALUES (102, 'Tschick', 'Herrndorf', 2010);
INSERT INTO Buch VALUES (103, 'Krabat', 'Preußler', 1971);
""")

con.commit()
con.close()

Füge dann die Bücher *Momo* (BuchID 104, Ende, 1973) und *Emil und die Detektive* (BuchID 105, Kästner, 1929) ein und gib zur Kontrolle alle Bücher aus.

#### Lösung

In [ ]:
import sqlite3

con = sqlite3.connect("bibliothek.db")
cur = con.cursor()

cur.execute("INSERT INTO Buch VALUES (104, 'Momo', 'Ende', 1973)")
cur.execute("INSERT INTO Buch VALUES (105, 'Emil und die Detektive', 'Kästner', 1929)")

con.commit()
con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("bibliothek.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Buch
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Ohne `commit()` wären die beiden Bücher nach `close()` nicht gespeichert.

### Aufgabe 2

Arbeite mit der Datenbank `bibliothek.db` aus Aufgabe 1 weiter.

1. Ändere den Autor von *Tschick* in `'Wolfgang Herrndorf'`.
2. Lösche alle Bücher, die vor 1950 erschienen sind.
3. Was wäre passiert, wenn du in Teil 2 das `WHERE` vergessen hättest?

#### Lösung

**1. und 2.** Die Bedingung in `WHERE` bestimmt, welche Datensätze betroffen sind:

In [ ]:
import sqlite3

con = sqlite3.connect("bibliothek.db")
cur = con.cursor()

cur.execute("UPDATE Buch SET Autor = 'Wolfgang Herrndorf' WHERE Titel = 'Tschick'")
cur.execute("DELETE FROM Buch WHERE Erscheinungsjahr < 1950")

con.commit()
con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("bibliothek.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Buch
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Gelöscht wurde nur *Emil und die Detektive* (1929).

**3.** `DELETE FROM Buch` ohne `WHERE` löscht **alle** Bücher. Die Tabelle selbst bleibt bestehen, sie ist dann aber leer.

## SQ 3 · Abfragen auf einer Tabelle

### Inhalte

Mit `SELECT` fragen wir Daten ab. Die Grundform ist:

```
SELECT Spalten
FROM Tabelle
WHERE Bedingung
```

- Hinter `SELECT` stehen die Spalten, die wir sehen wollen. `*` steht für **alle** Spalten. Die Auswahl von Spalten heißt **Projektion**.
- Mit `WHERE` wählen wir die Datensätze aus, die eine Bedingung erfüllen. Die Auswahl von Zeilen heißt **Selektion**.

Wie in SQ 1 holen wir das Ergebnis mit `cur.fetchall()` ab: eine Liste von Tupeln, jedes Tupel ist ein Datensatz.

**Beispiel (Projektion):** Gib die Namen aller Schülerinnen und Schüler aus.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname
    FROM Schueler
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel (Selektion):** Gib die Daten des Schülers mit der Nummer 121 aus.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Schueler
    WHERE SNr = 121
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

##### Bedingungen in WHERE

| Operator | Bedeutung | Beispiel |
|:--|:--|:--|
| `=`, `<>` | gleich, ungleich | `Kuerzel = 'SC'` |
| `<`, `<=`, `>`, `>=` | Vergleiche | `Stunden >= 4` |
| `AND`, `OR`, `NOT` | Bedingungen verknüpfen | `Kuerzel = 'SC' OR Kuerzel = 'AD'` |
| `LIKE` | Textmuster, `%` steht für beliebig viele Zeichen | `Name LIKE 'R%'` (beginnt mit R) |

Anders als in Python steht in SQL für „gleich“ ein **einfaches** `=`. Strings stehen in einfachen Hochkommata.

##### ORDER BY, DISTINCT und LIMIT

- `ORDER BY Spalte` **sortiert** das Ergebnis aufsteigend, `ORDER BY Spalte DESC` absteigend.
- `SELECT DISTINCT` entfernt **doppelte Zeilen** aus dem Ergebnis.
- `LIMIT n` begrenzt das Ergebnis auf die ersten n Datensätze. `LIMIT` steht immer am Ende, also nach `ORDER BY`.

**Beispiel:** Schulte unterrichtet zwei LKs, deshalb erscheint das Kürzel SC ohne DISTINCT doppelt.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Kuerzel
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT DISTINCT Kuerzel
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel:** Gib die ersten beiden Schülerinnen und Schüler in alphabetischer Reihenfolge aus.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname
    FROM Schueler
    ORDER BY Name
    LIMIT 2
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

### Aufgabe 1

Schreibe Abfragen für die Datenbank `lkwahl.db`:

1. Gib die Kürzel aller Lehrkräfte aus, alphabetisch sortiert.
2. Gib die Vornamen aller Schülerinnen und Schüler in umgekehrter alphabetischer Reihenfolge aus.
3. Gib die KursIDs aller Kurse von Schulte (Kürzel SC) oder Adam (Kürzel AD) aus.

#### Lösung

In [ ]:
# 1.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Kuerzel
    FROM Lehrer
    ORDER BY Kuerzel
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
# 2.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Vorname
    FROM Schueler
    ORDER BY Vorname DESC
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
# 3.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT KID
    FROM Kurse
    WHERE Kuerzel = 'SC' OR Kuerzel = 'AD'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

### Aufgabe 2

1. Gib Name und Vorname aller Schülerinnen und Schüler aus, deren Nachname mit `R` beginnt.
2. Gib aus, welche verschiedenen Stundenzahlen die Kurse haben.
3. Gib den Schüler bzw. die Schülerin mit der höchsten Schülernummer aus. Verwende dazu `ORDER BY` und `LIMIT`.

#### Lösung

In [ ]:
# 1.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname
    FROM Schueler
    WHERE Name LIKE 'R%'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
# 2.  Alle Kurse haben 5 Stunden, deshalb gibt es nur einen Wert.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT DISTINCT Stunden
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
# 3.  absteigend sortieren, dann nur den ersten Datensatz nehmen
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Schueler
    ORDER BY SNr DESC
    LIMIT 1
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

## SQ 4 · Abfragen über mehrere Tabellen

### Inhalte

Die Informationen sind nach der Normalisierung auf mehrere Tabellen verteilt. Um sie wieder zusammenzuführen, nennen wir hinter `FROM` **mehrere Tabellen** und verknüpfen sie in `WHERE` über **Fremdschlüssel und Primärschlüssel**. Das nennt man einen **Verbund** (Join).

So funktioniert es: `FROM Kurse, Lehrer` bildet zunächst **alle Kombinationen** aus einem Kurs und einer Lehrkraft, bei 5 Kursen und 4 Lehrkräften also 20 Zeilen. Die Bedingung `Kurse.Kuerzel = Lehrer.Kuerzel` behält davon nur die Zeilen, die wirklich zusammengehören.

Kommt ein Spaltenname in mehreren Tabellen vor, schreiben wir den **Tabellennamen** davor, z. B. `Kurse.Kuerzel`.

**Beispiel:** Gesucht sind alle Kurse von Schulte oder Adam, diesmal über den Namen der Lehrkraft.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT KID, Lehrer
    FROM Kurse, Lehrer
    WHERE Kurse.Kuerzel = Lehrer.Kuerzel
      AND (Lehrer = 'Schulte' OR Lehrer = 'Adam')
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Die Klammern um die OR-Bedingung sind wichtig, denn `AND` wird vor `OR` ausgewertet.

**Beispiel über vier Tabellen:** Gib für jede LK-Wahl den Namen des Schülers, die KursID und die Lehrkraft aus, sortiert nach Nachnamen. Für jede Verknüpfung zweier Tabellen brauchen wir eine Bedingung, bei vier Tabellen also drei.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname, LKWahl.KID, Lehrer
    FROM Schueler, LKWahl, Kurse, Lehrer
    WHERE Schueler.SNr = LKWahl.SNr
      AND LKWahl.KID = Kurse.KID
      AND Kurse.Kuerzel = Lehrer.Kuerzel
    ORDER BY Name
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Für eine übersichtlichere Ausgabe **entpacken** wir jedes Tupel in einzelne Variablen und benutzen einen f-String. `{name:10}` reserviert 10 Zeichen für den Namen, so entstehen gerade Spalten.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname, LKWahl.KID, Lehrer
    FROM Schueler, LKWahl, Kurse, Lehrer
    WHERE Schueler.SNr = LKWahl.SNr
      AND LKWahl.KID = Kurse.KID
      AND Kurse.Kuerzel = Lehrer.Kuerzel
    ORDER BY Name
""")

for name, vorname, kid, lehrer in cur.fetchall():          # unpacking
    print(f'{vorname:10} {name:10} {kid:4} {lehrer:10}')

con.close()

### Aufgabe 1

Welche Leistungskurse hat Lena Dreis gewählt? Die Abfrage soll mit dem Namen arbeiten, nicht mit der Schülernummer.

#### Lösung

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT KID
    FROM Schueler, LKWahl
    WHERE Schueler.SNr = LKWahl.SNr
      AND Name = 'Dreis' AND Vorname = 'Lena'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Die Schülernummer steht nur in *LKWahl*, der Name nur in *Schueler*. Deshalb brauchen wir beide Tabellen und eine Verknüpfungsbedingung.

### Aufgabe 2

Welche Schülerinnen und Schüler haben mindestens einen LK bei Schulte? Jeder Name soll nur einmal erscheinen, alphabetisch sortiert.

#### Lösung

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT DISTINCT Name, Vorname
    FROM Schueler, LKWahl, Kurse, Lehrer
    WHERE Schueler.SNr = LKWahl.SNr
      AND LKWahl.KID = Kurse.KID
      AND Kurse.Kuerzel = Lehrer.Kuerzel
      AND Lehrer = 'Schulte'
    ORDER BY Name
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Ohne `DISTINCT` erschiene Malte Riedberg zweimal, weil er zwei LKs bei Schulte hat (M1 und Ph1).

## SQ 5 · Verbund mit JOIN

### Inhalte

In SQ 4 haben wir die Tabellen hinter `FROM` aufgezählt und die **Verknüpfungsbedingung** in `WHERE` geschrieben. Dieselben Abfragen lassen sich auch mit `JOIN … ON` formulieren:

```
SELECT Spalten
FROM Tabelle1
JOIN Tabelle2 ON Verknüpfungsbedingung
WHERE weitere Bedingungen
```

- Hinter `JOIN` steht die Tabelle, die dazukommt.
- Hinter `ON` steht, **wie** sie mit den bisherigen Tabellen verknüpft ist, also Fremdschlüssel = Primärschlüssel.
- In `WHERE` stehen dann nur noch die Bedingungen, mit denen wir **Datensätze auswählen**.

Das Ergebnis ist dasselbe wie mit `WHERE`. Der Vorteil: Verknüpfung und Auswahl sind sauber getrennt. Bei jeder Tabelle sieht man sofort, über welche Spalten sie angebunden ist.

`JOIN` heißt ausführlich auch `INNER JOIN`. Im Ergebnis erscheinen nur Datensätze, die in beiden Tabellen einen passenden Partner haben.

##### Beispiel: zwei Tabellen

Gesucht sind alle Kurse von Schulte oder Adam (vgl. SQ 4).

| mit WHERE | mit JOIN |
|:--|:--|
| `SELECT KID, Lehrer` | `SELECT KID, Lehrer` |
| `FROM Kurse, Lehrer` | `FROM Kurse` |
| `WHERE Kurse.Kuerzel = Lehrer.Kuerzel` | `JOIN Lehrer ON Kurse.Kuerzel = Lehrer.Kuerzel` |
| `  AND (Lehrer = 'Schulte' OR Lehrer = 'Adam')` | `WHERE Lehrer = 'Schulte' OR Lehrer = 'Adam'` |

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT KID, Lehrer
    FROM Kurse
    JOIN Lehrer ON Kurse.Kuerzel = Lehrer.Kuerzel
    WHERE Lehrer = 'Schulte' OR Lehrer = 'Adam'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In der WHERE-Variante waren Klammern um die OR-Bedingung nötig, weil sie mit der Verknüpfungsbedingung durch `AND` verbunden war. Mit JOIN steht in `WHERE` nur noch die OR-Bedingung, die Klammern entfallen.

##### Beispiel: vier Tabellen

Gib für jede LK-Wahl den Namen des Schülers, die KursID und die Lehrkraft aus, sortiert nach Nachnamen (vgl. SQ 4). Für jede weitere Tabelle kommt eine Zeile mit `JOIN … ON` dazu.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname, LKWahl.KID, Lehrer
    FROM Schueler
    JOIN LKWahl ON Schueler.SNr = LKWahl.SNr
    JOIN Kurse ON LKWahl.KID = Kurse.KID
    JOIN Lehrer ON Kurse.Kuerzel = Lehrer.Kuerzel
    ORDER BY Name
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Die Abfrage hat keine `WHERE`-Klausel mehr, denn alle drei Bedingungen waren Verknüpfungsbedingungen. Man liest die Abfrage von oben nach unten wie einen Weg durch die Tabellen: Schueler → LKWahl → Kurse → Lehrer.

### Aufgabe 1

Welche Leistungskurse hat Lena Dreis gewählt? Schreibe die Abfrage aus SQ 4, Aufgabe 1, mit `JOIN`.

#### Lösung

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT KID
    FROM Schueler
    JOIN LKWahl ON Schueler.SNr = LKWahl.SNr
    WHERE Name = 'Dreis' AND Vorname = 'Lena'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Die Verknüpfung über die Schülernummer steht bei `JOIN`, in `WHERE` steht nur noch die Auswahl der Schülerin.

### Aufgabe 2

1. Welche Schülerinnen und Schüler haben mindestens einen LK bei Schulte? Jeder Name soll nur einmal erscheinen, alphabetisch sortiert. Schreibe die Abfrage aus SQ 4, Aufgabe 2, mit `JOIN`.
2. Führe die Abfrage `SELECT KID, Lehrer FROM Kurse, Lehrer` aus, also ohne Verknüpfungsbedingung. Wie viele Zeilen erhältst du? Erkläre.

#### Lösung

In [ ]:
# 1.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT DISTINCT Name, Vorname
    FROM Schueler
    JOIN LKWahl ON Schueler.SNr = LKWahl.SNr
    JOIN Kurse ON LKWahl.KID = Kurse.KID
    JOIN Lehrer ON Kurse.Kuerzel = Lehrer.Kuerzel
    WHERE Lehrer = 'Schulte'
    ORDER BY Name
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
# 2.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT KID, Lehrer
    FROM Kurse, Lehrer
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**2.** Man erhält **20 Zeilen**: Jeder der 5 Kurse wird mit jeder der 4 Lehrkräfte kombiniert, 5 · 4 = 20. Die meisten Kombinationen sind sinnlos, z. B. ('M1', 'Adam'), obwohl M1 von Schulte unterrichtet wird. Erst die Verknüpfungsbedingung, ob in `WHERE` oder hinter `ON`, behält nur die zusammengehörenden Zeilen.

Das passiert auch, wenn man in einer längeren Abfrage eine Verknüpfungsbedingung **vergisst**. Mit `JOIN … ON` fällt das leichter auf, weil zu jeder Tabelle ihr `ON` gehört.

## SQ 6 · Zählen und Summieren

### Inhalte

**Aggregatfunktionen** fassen viele Werte zu **einem einzigen** Wert zusammen.

| Funktion | Wirkung |
|:--|:--|
| `SUM(Spalte)` | addiert alle Werte einer Spalte |
| `COUNT(*)` | zählt die Datensätze im Ergebnis |
| `COUNT(DISTINCT Spalte)` | zählt die verschiedenen Werte einer Spalte |

Mit `WHERE` wählen wir vorher aus, welche Zeilen addiert oder gezählt werden.

**Beispiel:** Wie viele Wochenstunden LK-Unterricht gibt es insgesamt, und wie viele hat Schulte?

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT SUM(Stunden)
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT SUM(Stunden)
    FROM Kurse
    WHERE Kuerzel = 'SC'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel:** Wie viele Schülerinnen und Schüler sind gespeichert, und wie viele haben den Kurs Ph1 gewählt?

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM Schueler
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM LKWahl
    WHERE KID = 'Ph1'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel:** Wie viele verschiedene Lehrkräfte unterrichten einen LK? Mit `COUNT(DISTINCT …)` wird SC nur einmal gezählt (vgl. SQ 3).

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(DISTINCT Kuerzel)
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Das Ergebnis ist ein Tupel mit einem einzigen Wert, z. B. `(4,)`. Mit `cur.fetchall()[0][0]` erhalten wir die Zahl selbst und können sie in einen Satz einbauen:

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM Schueler
""")

anzahl = cur.fetchall()[0][0]
print(f'Es sind {anzahl} Schülerinnen und Schüler gespeichert.')

con.close()

### Aufgabe 1

1. Wie viele LK-Wahlen gibt es insgesamt?
2. Wie viele Kurse unterrichtet Schulte (Kürzel SC)?

#### Lösung

In [ ]:
# 1.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM LKWahl
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
# 2.
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM Kurse
    WHERE Kuerzel = 'SC'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

### Aufgabe 2

1. Wie viele Wochenstunden LK-Unterricht hat Malte Riedberg insgesamt?
2. Wie viele verschiedene Schülerinnen und Schüler haben einen LK bei Schulte? Gib das Ergebnis als Satz aus.

#### Lösung

In [ ]:
# 1.  Die Stunden stehen in Kurse, der Name in Schueler: Verbund über LKWahl
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT SUM(Stunden)
    FROM Schueler, LKWahl, Kurse
    WHERE Schueler.SNr = LKWahl.SNr
      AND LKWahl.KID = Kurse.KID
      AND Name = 'Riedberg' AND Vorname = 'Malte'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
# 2.  COUNT(DISTINCT ...), weil Malte Riedberg zwei LKs bei Schulte hat
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(DISTINCT LKWahl.SNr)
    FROM LKWahl, Kurse, Lehrer
    WHERE LKWahl.KID = Kurse.KID
      AND Kurse.Kuerzel = Lehrer.Kuerzel
      AND Lehrer = 'Schulte'
""")

anzahl = cur.fetchall()[0][0]
print(f'{anzahl} Schülerinnen und Schüler haben einen LK bei Schulte.')

con.close()